# Adversarial Data Augmentation: Fine-Tuning Against HotFlip

The text counterpart of `defenses/adversarial_training.ipynb` in the companion vision repo (PGD-AT
and TRADES on TrafficNet). The recipe is the same idea, expose the model to adversarial inputs
during training so it stops being fooled by them, adapted to a discrete input space: instead of
generating a perturbed image on the fly inside the training loop, we generate a batch of adversarial
sentences with `attacks/whitebox/01_HotFlip.ipynb`'s attack once, label them with the ground truth (not the
flipped prediction), and mix them into the fine-tuning data.

We then compare, before and after this augmented fine-tuning: clean accuracy, and Attack Success
Rate for both HotFlip (white-box) and Synonym Substitution (black-box) on a held-out set of
sentences never used for augmentation, so the evaluation measures generalized robustness, not just
memorization of the specific adversarial examples seen during training.


In [1]:
import copy
import torch
import torch.nn.functional as F
import nltk
from datasets import load_dataset
from nltk.corpus import wordnet, stopwords
from transformers import AutoTokenizer, AutoModelForSequenceClassification

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords', quiet=True)

STOPWORDS = set(stopwords.words('english'))
torch.manual_seed(0)

### Setup: Model, Data
Same target as the two attack notebooks, `distilbert-base-uncased-finetuned-sst-2-english`, and
the same dataset, the real SST-2 validation split (used here as a stand-in fine-tuning/eval corpus
since the original training set is much larger than needed for this demonstration). We keep a
`baseline_model` untouched and fine-tune a separate `augmented_model` copy, so both can be compared
against the exact same attacks afterwards.

In [2]:
MODEL_NAME = 'distilbert-base-uncased-finetuned-sst-2-english'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
baseline_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
baseline_model.eval()
LABELS = baseline_model.config.id2label
SPECIAL_TOKEN_IDS = set(tokenizer.all_special_ids)
embedding_matrix = baseline_model.get_input_embeddings().weight

sst2 = load_dataset('stanfordnlp/sst2', split='validation')
N_AUGMENT = 150
N_EVAL = 40
augment_pool = sst2.select(range(N_AUGMENT))
eval_pool = sst2.select(range(N_AUGMENT, N_AUGMENT + N_EVAL))
print(f"Augmentation pool: {len(augment_pool)} sentences, held-out eval pool: {len(eval_pool)} sentences")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Augmentation pool: 150 sentences, held-out eval pool: 40 sentences


### Step 1: Generate Adversarial Augmentation Examples
HotFlip against every sentence in the augmentation pool, using `baseline_model` so the adversarial
examples reflect the vulnerabilities of the model we are about to fine-tune. Each perturbed sentence
is paired with its **ground-truth** label (from the dataset, not the model's flipped prediction),
the whole point is teaching the model "this still means positive/negative even though a few words
changed", not teaching it to imitate its own mistake.

In [3]:
def predict(model, text):
    inputs = tokenizer(text, return_tensors='pt')
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = F.softmax(logits, dim=-1)[0]
    label_idx = int(torch.argmax(probs))
    return LABELS[label_idx], float(probs[label_idx]), label_idx

def hotflip_step(model, embed_matrix, input_ids, true_label_idx):
    embeds = embed_matrix[input_ids].clone().detach().unsqueeze(0)
    embeds.requires_grad_(True)
    logits = model(inputs_embeds=embeds).logits
    loss = F.cross_entropy(logits, torch.tensor([true_label_idx]))
    loss.backward()
    grad = embeds.grad[0]

    best_score, best_pos, best_candidate = float('-inf'), None, None
    for pos in range(len(input_ids)):
        if input_ids[pos].item() in SPECIAL_TOKEN_IDS:
            continue
        current_embed = embed_matrix[input_ids[pos]]
        scores = (embed_matrix - current_embed) @ grad[pos]
        candidate_id = int(torch.argmax(scores))
        if scores[candidate_id] > best_score:
            best_score = float(scores[candidate_id])
            best_pos, best_candidate = pos, candidate_id
    return best_pos, best_candidate

def hotflip_attack(model, text, true_label_idx, max_flips=3):
    input_ids = tokenizer(text, return_tensors='pt')['input_ids'][0]
    for _ in range(max_flips):
        pos, candidate_id = hotflip_step(model, embedding_matrix, input_ids, true_label_idx)
        input_ids = input_ids.clone()
        input_ids[pos] = candidate_id
        adv_text = tokenizer.decode(input_ids, skip_special_tokens=True)
        _, _, current_label_idx = predict(model, adv_text)
        if current_label_idx != true_label_idx:
            break
    return tokenizer.decode(input_ids, skip_special_tokens=True)

In [4]:
augmented_pairs = []
for example in augment_pool:
    text, true_label_idx = example['sentence'].strip(), example['label']
    adv_text = hotflip_attack(baseline_model, text, true_label_idx, max_flips=3)
    augmented_pairs.append((adv_text, true_label_idx))

print(f"Generated {len(augmented_pairs)} adversarial augmentation examples")
for adv_text, label_idx in augmented_pairs[:3]:
    print(f"  [{LABELS[label_idx]}] {adv_text}")

C:\Users\frang\AppData\Local\Temp\ipykernel_20856\1310312493.py:25: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:821.)
  best_score = float(scores[candidate_id])


Generated 150 adversarial augmentation examples
  [POSITIVE] it ' s a waste and often affecting journey.
  [NEGATIVE] unflinchingly bleak and intriguing
  [POSITIVE] allows us to hope that nolan is poised embark a major career as a mess yet inventive filmmaker.


### Step 2: Fine-Tune on Original + Adversarial Examples
A plain supervised fine-tuning loop (cross-entropy, AdamW, small learning rate since the model is
already well-trained) over the union of the clean augmentation pool and its adversarial
counterparts, so the model sees both the original phrasing and the perturbed one for every example,
the text equivalent of training on clean-plus-perturbed image batches in PGD adversarial training.

In [5]:
augmented_model = copy.deepcopy(baseline_model)
augmented_model.train()
optimizer = torch.optim.AdamW(augmented_model.parameters(), lr=2e-5)

clean_pairs = [(example['sentence'].strip(), example['label']) for example in augment_pool]
training_pairs = clean_pairs + augmented_pairs
EPOCHS = 2

for epoch in range(EPOCHS):
    total_loss = 0.0
    for text, label_idx in training_pairs:
        inputs = tokenizer(text, return_tensors='pt')
        optimizer.zero_grad()
        logits = augmented_model(**inputs).logits
        loss = F.cross_entropy(logits, torch.tensor([label_idx]))
        loss.backward()
        optimizer.step()
        total_loss += float(loss)
    print(f"Epoch {epoch + 1}/{EPOCHS}, mean loss: {total_loss / len(training_pairs):.4f}")

augmented_model.eval()

Epoch 1/2, mean loss: 0.8037


Epoch 2/2, mean loss: 0.3751


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

### Step 3: Evaluate Clean Accuracy, Before and After
On the held-out eval pool (never seen during augmentation or fine-tuning).

In [6]:
def clean_accuracy(model, pool):
    correct = 0
    for example in pool:
        _, _, pred_idx = predict(model, example['sentence'].strip())
        correct += int(pred_idx == example['label'])
    return correct / len(pool)

baseline_clean_acc = clean_accuracy(baseline_model, eval_pool)
augmented_clean_acc = clean_accuracy(augmented_model, eval_pool)
print(f"Clean accuracy, baseline:  {baseline_clean_acc * 100:.1f}%")
print(f"Clean accuracy, augmented: {augmented_clean_acc * 100:.1f}%")

Clean accuracy, baseline:  90.0%
Clean accuracy, augmented: 82.5%


### Step 4: Evaluate Robustness, Before and After
Attack Success Rate of HotFlip and Synonym Substitution against both models, restricted to
held-out sentences each model still classifies correctly (the standard "robust accuracy"
convention also used throughout the vision repo's Report Card): an attack that flips an
already-wrong prediction is not a meaningful success.

In [7]:
POS_MAP = {'NN': wordnet.NOUN, 'VB': wordnet.VERB, 'JJ': wordnet.ADJ, 'RB': wordnet.ADV}

def get_synonyms(word, pos_tag, max_candidates=8):
    wn_pos = POS_MAP.get(pos_tag[:2])
    if wn_pos is None:
        return []
    synonyms = set()
    for syn in wordnet.synsets(word, pos=wn_pos):
        for lemma in syn.lemmas():
            candidate = lemma.name().replace('_', ' ')
            if candidate.lower() != word.lower():
                synonyms.add(candidate)
    return list(synonyms)[:max_candidates]

def synonym_attack(model, text, max_words_to_replace=10):
    words = text.split()
    pos_tags = [tag for _, tag in nltk.pos_tag(words)]
    _, _, true_label_idx = predict(model, text)

    with torch.no_grad():
        base_inputs = tokenizer(text, return_tensors='pt')
        base_prob = float(F.softmax(model(**base_inputs).logits, dim=-1)[0, true_label_idx])
    importances = []
    for i in range(len(words)):
        if words[i].lower() in STOPWORDS:
            importances.append((i, -1.0))
            continue
        without_word = ' '.join(words[:i] + words[i + 1:])
        with torch.no_grad():
            inputs = tokenizer(without_word, return_tensors='pt')
            prob = float(F.softmax(model(**inputs).logits, dim=-1)[0, true_label_idx])
        importances.append((i, base_prob - prob))
    order = [i for i, _ in sorted(importances, key=lambda x: x[1], reverse=True)]

    for count, i in enumerate(order):
        if count >= max_words_to_replace:
            break
        best_word, best_prob = words[i], None
        for candidate in get_synonyms(words[i], pos_tags[i]):
            trial_words = words[:i] + [candidate] + words[i + 1:]
            with torch.no_grad():
                inputs = tokenizer(' '.join(trial_words), return_tensors='pt')
                trial_prob = float(F.softmax(model(**inputs).logits, dim=-1)[0, true_label_idx])
            if best_prob is None or trial_prob < best_prob:
                best_word, best_prob = candidate, trial_prob
        if best_word != words[i]:
            words[i] = best_word
        _, _, current_label_idx = predict(model, ' '.join(words))
        if current_label_idx != true_label_idx:
            break
    return ' '.join(words)

In [8]:
def attack_success_rate(model, pool, attack_fn):
    attacked, successes = 0, 0
    for example in pool:
        text, true_label_idx = example['sentence'].strip(), example['label']
        _, _, pred_idx = predict(model, text)
        if pred_idx != true_label_idx:
            continue  # only attack sentences the model already gets right
        attacked += 1
        adv_text = attack_fn(model, text, true_label_idx)
        _, _, adv_pred_idx = predict(model, adv_text)
        successes += int(adv_pred_idx != true_label_idx)
    return successes / attacked if attacked else float('nan'), attacked

def hotflip_attack_fn(model, text, true_label_idx):
    return hotflip_attack(model, text, true_label_idx, max_flips=5)

results = {}
for name, model in [('Baseline', baseline_model), ('Augmented', augmented_model)]:
    hf_asr, hf_n = attack_success_rate(model, eval_pool, hotflip_attack_fn)
    ss_asr, ss_n = attack_success_rate(model, eval_pool, synonym_attack)
    results[name] = {'hotflip_asr': hf_asr, 'hotflip_n': hf_n, 'synonym_asr': ss_asr, 'synonym_n': ss_n}
    print(f"{name}: HotFlip ASR {hf_asr * 100:.1f}% (n={hf_n}), Synonym ASR {ss_asr * 100:.1f}% (n={ss_n})")

Baseline: HotFlip ASR 83.3% (n=36), Synonym ASR 16.7% (n=36)


Augmented: HotFlip ASR 57.6% (n=33), Synonym ASR 18.2% (n=33)


### Summary
Clean accuracy and Attack Success Rate, baseline vs. augmented model, side by side.

In [9]:
print(f"{'Model':<12} {'Clean Acc':>10} {'HotFlip ASR':>13} {'Synonym ASR':>13}")
print(f"{'Baseline':<12} {baseline_clean_acc * 100:>9.1f}% {results['Baseline']['hotflip_asr'] * 100:>12.1f}% {results['Baseline']['synonym_asr'] * 100:>12.1f}%")
print(f"{'Augmented':<12} {augmented_clean_acc * 100:>9.1f}% {results['Augmented']['hotflip_asr'] * 100:>12.1f}% {results['Augmented']['synonym_asr'] * 100:>12.1f}%")

Model         Clean Acc   HotFlip ASR   Synonym ASR
Baseline          90.0%         83.3%         16.7%
Augmented         82.5%         57.6%         18.2%
